# 介紹

使用 `pyautogui` 偵測 Chrome 小恐龍前方的障礙物並自動跳躍。

小恐龍是在 Chrome 中使用 `chrome://dino` 啟動。執行前請先開啟遊戲，並將瀏覽器視窗放在螢幕上。

偵測座標以螢幕左上角為原點；目前依恐龍右緣約 x=62、地面線約 y=341 設定偵測區。若遊戲視窗移動或縮放，請調整 `DETECTION_REGION`。按 Esc 可停止程式，或在 Jupyter 中按停止按鈕。

程式只截取恐龍前方的一小塊畫面，並以亮度門檻偵測深色像素，減少全螢幕截圖的負擔，也避免要求顏色完全相同造成漏判。

In [ ]:
import time
import pyautogui as pyg

# 偵測區域格式：(螢幕 x, 螢幕 y, 寬度, 高度)，請依遊戲視窗位置調整。
# 區域底部不要包含地面線，否則地面可能被誤認為障礙物。
DETECTION_REGION = (63, 311, 120, 25)
DARK_PIXEL_THRESHOLD = 150  # (83, 83, 83) 障礙物會通過；(255, 255, 255) 白底會忽略
MIN_DARK_PIXELS = 3  # 忽略少量雜訊像素
LOOP_DELAY = 0.015

def obstacle_is_near(image):
    """檢查偵測區域內是否有足夠多的深色像素。"""
    dark_pixels = 0
    for red, green, blue in image.convert('RGB').getdata():
        if (red + green + blue) / 3 < DARK_PIXEL_THRESHOLD:
            dark_pixels += 1
            if dark_pixels >= MIN_DARK_PIXELS:
                return True
    return False

pyg.PAUSE = 0  # 移除 pyautogui 預設的額外延遲
pyg.sleep(2)  # 留時間切換到遊戲視窗
pyg.press('space')  # 開始遊戲

was_obstacle_near = False
try:
    while not pyg.is_pressed('esc'):
        screen = pyg.screenshot(region=DETECTION_REGION)
        obstacle_near = obstacle_is_near(screen)

        # 障礙物剛進入偵測區時才跳一次，避免每一幀重複觸發。
        if obstacle_near and not was_obstacle_near:
            pyg.press('space')
        was_obstacle_near = obstacle_near
        time.sleep(LOOP_DELAY)
finally:
    pyg.keyUp('space')  # 停止時確保空白鍵已放開
    print('恐龍機器人已停止。')

In [ ]:
# 檢查螢幕尺寸與指定座標的像素顏色，可用來校準偵測區域。
import pyautogui as pyg

sample_point = (103, 420)  # 改成想檢查的螢幕座標
screen = pyg.screenshot()
print(f'螢幕大小：{screen.size}')
print(f'{sample_point} 的 RGB 顏色：{screen.getpixel(sample_point)}')